In [2]:
import pandas as pd

# read OxCGRT Australian policy data
df = pd.read_csv("../data/raw/OxCGRT_AUS_latest.csv")

# keep the colunms
col_subsets = ["RegionName", "RegionCode", "Date", "H6M_Facial Coverings"]
df = df[col_subsets]

# convert date format
df.index = pd.to_datetime(df["Date"], format="%Y%m%d")

# 14-day rolling average
rolling_days = 14
df_rolling = df[["RegionName", "H6M_Facial Coverings"]].groupby("RegionName").rolling(window=rolling_days).mean()

# Find the location where the mandate condition is first met
mandate_limit = 3
df_mandates = df_rolling[df_rolling["H6M_Facial Coverings"] >= mandate_limit].groupby("RegionName").head(1)

# save the mandate start datas
df_mandates.to_csv("../data/processed/mandate_start_dates.csv")
print(df_mandates)

                                         H6M_Facial Coverings
RegionName                   Date                            
Australian Capital Territory 2021-08-18              3.000000
New South Wales              2021-07-09              3.000000
Northern Territory           2021-11-21              3.000000
Queensland                   2021-01-17              3.142857
South Australia              2021-07-26              3.000000
Tasmania                     2021-10-21              3.000000
Victoria                     2020-07-21              3.000000
Western Australia            2021-02-08              3.000000


In [1]:
import pandas as pd

# determine whether it is after the mask mandate begins
def within_mandate(row):
    state = row["state"]
    endtime = pd.to_datetime(row["endtime"])

    if states_date[state] <= endtime:
        return 1
    return 0

# read cleaned_date
df = pd.read_csv("../data/processed/cleaned_data.csv", keep_default_na=False)

# read the generated mandate start date
mandate_df = pd.read_csv("../data/processed/mandate_start_dates.csv")

states_date = {}
for state, date in zip(mandate_df["RegionName"], mandate_df["Date"]):
    states_date[state] = pd.to_datetime(date)

# create variables that are within the mandate period or not
df["within_mandate_period"] = df.apply(within_mandate, axis=1)

# classification variables (require dummy encoding)
categorical_cols = [
    "state", "gender", "i9_health", "employment_status", "i11_health",
    "WCRex1", "WCRex2", "PHQ4_1", "PHQ4_2", "PHQ4_3", "PHQ4_4",
    "d1_comorbidities"
]

# convert to dummy variables
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# save preprocessing data
df.to_csv("../data/processed/cleaned_data_preprocessing.csv", index=False)